In [ ]:

import asyncio
import re
import sys



class Logable:
    def __init__(self, name, verbose=True):
        self.name = name
        self.verbose = verbose

    def log(self, message):
        if self.verbose:
            print(f"[{self.name}] {message}")



class Pic(Logable):
    """File FIFO contenant les commandes à préparer."""

    def __init__(self, name="pic", verbose=True):
        super().__init__(name, verbose)
        self.queue = asyncio.Queue()

    async def embrocher(self, postit):
        await self.queue.put(postit)
        self.log(f"post-it ajouté : {postit}")

    async def liberer(self):
        return await self.queue.get()

    def empty(self):
        return self.queue.empty()


class Bar(Logable):
    """File FIFO contenant les commandes prêtes à servir."""

    def __init__(self, name, verbose=True):
        super().__init__(name, verbose)
        self.queue = asyncio.Queue()

    async def recevoir(self, commande):
        await self.queue.put(commande)
        self.log(f"commande déposée : {commande}")

    async def evacuer(self):
        return await self.queue.get()


class Clients:

    def __init__(self, fname):
        self.commandes = []

        with open(fname, "r", encoding="utf-8") as fichier:
            for ligne in fichier:
                resultat = re.match(r"(\d+)\s+(.*)", ligne.strip())

                if resultat:
                    attente = int(resultat.group(1))
                    consommations = resultat.group(2).split(",")
                    self.commandes.append((attente, consommations))

    async def run(self, serveurs, pic, clients_done):
        """
        Les clients transmettent leurs commandes.
        Les serveurs sont sollicités à tour de rôle.
        """

        for i, (attente, commande) in enumerate(self.commandes):

            # Attente avant l'arrivée de la prochaine commande
            await asyncio.sleep(attente)

            serveur = serveurs[i % len(serveurs)]

            serveur.log(f"nouvelle commande : {commande}")

            # Le post-it indique le serveur destinataire
            await pic.embrocher((serveur, commande))

        # Indique que tous les clients ont transmis leurs commandes
        clients_done.set()



class Employe(Logable):

    def __init__(self, name, pic, bar, productivite=1):
        super().__init__(name)

        self.pic = pic
        self.bar = bar
        self.productivite = productivite



class Serveur(Employe):

    def __init__(self, name, pic, bar, productivite, service_lock):
        super().__init__(name, pic, bar, productivite)

        # Verrou partagé avec les autres employés
        self.service_lock = service_lock

    async def servir(self, commande):
        """
        Le serveur sert une commande complète.
        Il conserve le verrou pendant tout le service.
        """

        async with self.service_lock:

            self.log(f"j'apporte la commande {commande}")

            for consommation in commande:
                self.log(f"je sers {consommation}")

                await asyncio.sleep(self.productivite)

    async def run(self):
        """Le serveur récupère et sert les commandes de son bar."""

        self.log("prêt pour le service")

        while True:

            commande = await self.bar.evacuer()

            # None indique la fin du travail
            if commande is None:
                self.log("fin du service")
                break

            await self.servir(commande)



class Bariste(Employe):

    def __init__(
        self,
        name,
        pic,
        serveurs,
        productivite,
        service_lock,
        clients_done
    ):
        super().__init__(name, pic, None, productivite)

        self.serveurs = serveurs
        self.service_lock = service_lock
        self.clients_done = clients_done

    async def preparer(self, commande):
        """Prépare les cocktails d'une commande."""

        self.log(f"je commence la fabrication de {commande}")

        for consommation in commande:

            self.log(f"je prépare {consommation}")

            await asyncio.sleep(self.productivite)

        self.log(f"la commande {commande} est prête")

    async def servir_directement(self, commande):
        """
        Extension 3 :
        le bariste sert lui-même la commande lorsque le Pic est vide.
        Il ne passe donc pas par un post-it ni par le bar d'un serveur.
        """

        async with self.service_lock:

            self.log(f"le Pic est vide : je prends en charge le service de {commande}")

            for consommation in commande:

                self.log(f"je sers directement {consommation}")

                await asyncio.sleep(self.productivite)

    async def run(self):
        """Le bariste prépare les commandes et peut aider au service."""

        self.log("prêt pour le service")

        while True:

            # Si le Pic est vide, on vérifie si les clients ont terminé
            if self.pic.empty():

                if self.clients_done.is_set():
                    break

                # On laisse la boucle d'événements travailler
                await asyncio.sleep(0.01)
                continue

            # Récupération du prochain post-it
            serveur, commande = await self.pic.liberer()

            # Préparation de la commande
            await self.preparer(commande)

            # EXTENSION 3 :
            # Si le Pic est vide après la préparation,
            # le bariste peut servir directement.
            if self.pic.empty():

                await self.servir_directement(commande)

            else:

                # Sinon, il transmet la commande au serveur concerné
                await serveur.bar.recevoir(commande)

        # Le bariste a terminé toutes les préparations.
        # Il informe chaque serveur qu'il n'y a plus de commande.
        for serveur in self.serveurs:
            await serveur.bar.recevoir(None)

        self.log("fin du travail")



async def main(fname):

    # Accessoire partagé par les deux serveurs
    pic = Pic()

    # Chaque serveur possède son propre bar
    bar_alice = Bar("bar_alice")
    bar_charlie = Bar("bar_charlie")

    # Un seul verrou pour coordonner tous les services
    service_lock = asyncio.Lock()

    # Événement indiquant que les clients ont fini
    clients_done = asyncio.Event()

    # Création des serveurs
    alice = Serveur(
        "alice",
        pic,
        bar_alice,
        productivite=1,
        service_lock=service_lock
    )

    charlie = Serveur(
        "charlie",
        pic,
        bar_charlie,
        productivite=3,
        service_lock=service_lock
    )

    serveurs = [alice, charlie]

    # Création du bariste
    bob = Bariste(
        "bob",
        pic,
        serveurs,
        productivite=2,
        service_lock=service_lock,
        clients_done=clients_done
    )

    # Création des clients
    clients = Clients(fname)

    # Lancement simultané des tâches
    await asyncio.gather(
        clients.run(serveurs, pic, clients_done),
        bob.run(),
        alice.run(),
        charlie.run()
    )



await main("lundi")


RuntimeError: asyncio.run() cannot be called from a running event loop